In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_wavelet_ica_frequency_channel import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 04 — ICA on wavelet tensors (frequency channel)

Observations are subject x time, features are frequency x channel.

**The method is documented in
[`docs/analyses/04-wavelet-ica-analysis/`](../../docs/analyses/04-wavelet-ica-analysis/README.md)** — this
notebook does not restate it.

**What this notebook is for.** Choosing parameters and checking the workflow cheaply
before committing a node to it. It calls `run()` from
[`scripts/run_wavelet_ica_frequency_channel.py`](../../scripts/run_wavelet_ica_frequency_channel.py) — the same function the PBS job
calls — at reduced scale, then shows what it wrote.

## Configuration

In [ ]:
# The decomposition to run. Small on purpose -- the full-size run is the job's.
EXPERIMENT = "assr"  # or "psilo_music"
CONDITION = "Placebo"
N_PCA = 5
BAND = None

# IVA ranks components by how much the subjects agree about them; these pick from each
# end of that ranking and together may not exceed N_PCA.
N_TOP = 3
N_BOTTOM = 2

# Subset every axis before the wavelet transform. This is what makes a pass take
# seconds; the same three limits exist on every stage's script.
N_SUBJECTS = 6
N_CHANNELS = 32
N_TIMES = 3000  # samples (~12 s at 250 Hz)

OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "out" / "frequency_channel"
)
PLOTS_DIR = OUT_DIR / "plots"

## Run

The same function the PBS job calls. Keep the scale small here: the full-size pass is what
[`jobs/metacentrum/04-wavelet-ica-analysis/`](../../jobs/metacentrum/04-wavelet-ica-analysis) is for.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--experiment",
        EXPERIMENT,
        "--condition",
        CONDITION,
        "--n_pca",
        str(N_PCA),
        "--n_ica",
        str(N_PCA),
        *(["--band", BAND] if BAND else []),
        "--n_subjects",
        str(N_SUBJECTS),
        "--n_channels",
        str(N_CHANNELS),
        "--n_times",
        str(N_TIMES),
        "--n_ica",
        str(N_PCA),
        "--save_dir",
        str(PLOTS_DIR),
    ]
)
run(args)

## Figures

Written by the shared plotters in `src/visualization/`, which the CLI uses too --
so what is shown here is what the job produces.

In [ ]:
for png in sorted(PLOTS_DIR.rglob("*.png")):
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))